In [ ]:
#| default_exp docsprocs

# docsprocs

> A docs-build notebook processor that color-codes cells by boopiter type, so the rendered site echoes the app's colored left bars.

Registered via `doc_procs` in `pyproject.toml [tool.nbdev]`, `color_cells` runs over every notebook during the docs build (before Quarto renders). It wraps each markdown/raw cell's source in a Quarto fenced div classed by its boopiter cell type (`.boop-note` green, `.boop-prompt` red, `.boop-raw` orange); `styles.css` turns those into the colored left bar. Code cells already carry `.cell-code`, so they're colored in CSS alone. The page's H1 title cell is left untouched so Quarto's title handling isn't disturbed.

In [ ]:
#| export
import re

# a Prompt+Reply markdown cell (solveit encoding) carries this separator -- see serialize.py
_SEP_RE = re.compile(r'##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_[0-9a-f]+ -->')

def color_cells(cell):
    "nbdev docs processor: wrap a markdown/raw cell's source in a Quarto fenced div classed by its boopiter cell type, so styles.css can draw boopiter's colored left bar (green note / red prompt / orange raw). Code cells carry `.cell-code` and are colored via CSS; the page-title (H1) cell is left alone."
    t = cell.get('cell_type'); src = cell.get('source') or ''
    if not src.strip(): return
    first = src.lstrip().splitlines()[0]
    if t == 'raw': cls = 'boop-raw'
    elif t == 'markdown':
        if first.startswith('# '): return          # leave the page-title (H1) cell alone
        cls = 'boop-prompt' if _SEP_RE.search(src) else 'boop-note'
    else: return                                    # code cells are handled in CSS
    cell['source'] = f'::: {{.boopcell .{cls}}}\n{src}\n:::'
